In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor


In [2]:
train = MNIST("/mnt/nishant/datasets", train=True, download=True, transform=ToTensor())
test = MNIST("/mnt/nishant/datasets", train=False, download=True, transform=ToTensor())

train_dataloader = DataLoader(train, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test, batch_size=512)

input_dim = 28 * 28
mlp_hidden_layers = [256, 128, 64, 32, 16, 10]
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_dataloader:
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

    if epoch % 2 == 0:
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_dataloader:
                pred = model(x).argmax(1)
                correct += (pred == y).sum().item()
                total += y.numel()
        print(f"epoch {epoch} acc {correct / total:.4f}")


epoch 2 acc 0.9453
epoch 4 acc 0.9638


In [3]:
# after training
wb = [
    {"W": m.weight.detach().cpu().clone(), "b": m.bias.detach().cpu().clone()}
    for m in model
    if isinstance(m, nn.Linear)
]

torch.save(wb, "mlp_wb.pt")  # save
# wb = torch.load("mlp_wb.pt")  # load later


#### Create A_l for each layer

In [4]:
# W: (n_l, n_{l-1}),
# b: (n_l,), l in {1..L}


def make_A(weights, bias, ell, num_layers):
    nl, nprev = weights.shape
    k = num_layers - ell  # remaining layers after ell
    top_left = torch.cat([weights, bias[:, None]], dim=1)  # [W_l  b_l] -> (nl, nprev+1)
    if k == 0:
        return top_left

    # shape: nl x L-ell
    Z_top_right = torch.zeros(nl, k, dtype=weights.dtype, device=weights.device)

    # shape: L-ell x nprev+1
    Z_bottom_left = torch.zeros(
        k, nprev + 1, dtype=weights.dtype, device=weights.device
    )

    # I_{L-l}
    eye_bottom_right = torch.eye(k, dtype=weights.dtype, device=weights.device)

    return torch.cat(
        [
            torch.cat([top_left, Z_top_right], 1),
            torch.cat([Z_bottom_left, eye_bottom_right], 1),
        ],
        0,
    )


def make_K_list(
    A,
    input_dim,
    hidden_layers,
    delta,
):
    L = len(A)
    embedding_dim = L + max([input_dim] + list(hidden_layers))  # include n0
    Kl = []
    for Al in A:
        U, s, Vh = torch.linalg.svd(Al, full_matrices=True)
        p, q = Al.shape
        r = min(p, q)

        U_t = torch.block_diag(
            U, torch.eye(embedding_dim - p, device=Al.device, dtype=Al.dtype)
        )
        V_t = torch.block_diag(
            Vh.T, torch.eye(embedding_dim - q, device=Al.device, dtype=Al.dtype)
        )

        S_t = torch.eye(embedding_dim, device=Al.device, dtype=Al.dtype) * delta
        S_t[:r, :r] = torch.diag(s)

        Kl.append(U_t @ S_t @ V_t.T)

    return Kl, embedding_dim


# x: (n0,) or (n0,1);
# Kl: list of N×N; n_list: [n1,...,nL]
def forward_manual(x, Kl, n0, n_list):
    L, N = len(Kl), Kl[0].shape[0]
    a = x.view(n0, 1).to(device=Kl[0].device, dtype=Kl[0].dtype)

    for l in range(1, L + 1):
        ones = torch.ones(L - l + 1, 1, device=a.device, dtype=a.dtype)
        zeros = torch.zeros(
            N - a.shape[0] - (L - l + 1), 1, device=a.device, dtype=a.dtype
        )
        y_in = torch.cat([a, ones, zeros], dim=0)  # (N,1)

        y_out = Kl[l - 1] @ y_in  # (N,1)
        z = y_out[: n_list[l - 1]]  # (n_l,1)

        a = z if l == L else torch.relu(z)

    return a


# returns list [t_1,...,t_{L-1}], each (p_l,1) where p_l = n_l + (L-l)
def get_t_list(model, img, num_layers):
    targets = []
    a = img.view(1, -1)  # (1,n0)

    counter = 0
    for m in model:
        if isinstance(m, nn.Linear):
            counter += 1
            z = m(a)  # (1,n_l)
            if counter < num_layers:  # no t_L
                a = torch.relu(z)
                ones = torch.ones(
                    1, num_layers - counter, device=a.device, dtype=a.dtype
                )
                targets.append(torch.cat([a, ones], dim=1).T)  # (p_l,1)
    return targets

In [5]:
def rot_u_to_w(u, w, eps=1e-12):
    # u,w: (p,1) unit vectors -> R in SO(p) with R@u = w
    p = u.size(0)
    I = torch.eye(p, device=u.device, dtype=u.dtype)
    c = (u.T @ w).clamp(-1.0, 1.0)  # (1,1)

    if (c > 1 - 1e-7).item():
        return I
    if (c < -1 + 1e-7).item():  # 180° case: pick any v ⟂ u
        e = torch.zeros_like(u)
        e[torch.argmin(u.abs())] = 1
        v = e - u * (u.T @ e)
        v = v / (v.norm() + eps)
        return I - 2 * (u @ u.T + v @ v.T)

    v = w - c * u
    v = v / (v.norm() + eps)
    s = torch.sqrt(1 - c * c)
    return I + (c - 1) * (u @ u.T + v @ v.T) + s * (v @ u.T - u @ v.T)


def compute_RSO_and_v(x, Kl, n0, n_list, t_list, eps=1e-12):
    L, N = len(Kl), Kl[0].shape[0]
    a = x.view(n0, 1).to(device=Kl[0].device, dtype=Kl[0].dtype)

    RSO, v = [], []
    for l in range(1, L + 1):
        ones = torch.ones(L - l + 1, 1, device=a.device, dtype=a.dtype)
        zeros = torch.zeros(
            N - a.size(0) - (L - l + 1), 1, device=a.device, dtype=a.dtype
        )
        y_out = Kl[l - 1] @ torch.cat([a, ones, zeros], 0)  # (N,1)

        if l < L:
            p = n_list[l - 1] + (L - l)
            pre = y_out[:p]
            t = t_list[l - 1].to(device=pre.device, dtype=pre.dtype)

            np, nt = pre.norm(), t.norm()
            RSO.append(rot_u_to_w(pre / (np + eps), t / (nt + eps), eps=eps))
            v.append(nt / (np + eps))

        z = y_out[: n_list[l - 1]]
        a = z if l == L else torch.relu(z)

    return RSO, torch.stack(v)  # RSO: list of (p_l,p_l), v: (L-1,)

In [6]:
# uses (RSO, v) to replace ReLU; no relu anywhere
def forward_no_relu(x, Kl, RSO, v, n0, n_list):
    L, N = len(Kl), Kl[0].shape[0]
    a = x.view(n0, 1).to(device=Kl[0].device, dtype=Kl[0].dtype)

    for l in range(1, L + 1):
        ones = torch.ones(L - l + 1, 1, device=a.device, dtype=a.dtype)
        zeros = torch.zeros(
            N - a.size(0) - (L - l + 1), 1, device=a.device, dtype=a.dtype
        )
        y_out = Kl[l - 1] @ torch.cat([a, ones, zeros], 0)  # (N,1)

        if l < L:
            p = n_list[l - 1] + (L - l)
            pre = y_out[:p]  # (p_l,1)
            a = (v[l - 1] * (RSO[l - 1] @ pre))[: n_list[l - 1]]  # next a is (n_l,1)
        else:
            return y_out[: n_list[-1]]  # (10,1)


In [7]:
num_layers = len(wb)

A_matrix_list = [
    make_A(wb[i]["W"], wb[i]["b"], i + 1, num_layers) for i in range(num_layers)
]

svds_of_A_matrix = [torch.linalg.svd(a, full_matrices=False) for a in A_matrix_list]

K_matrix_list, embedding_dim = make_K_list(
    A_matrix_list,
    input_dim=input_dim,
    hidden_layers=mlp_hidden_layers,
    delta=0.1,
)

model.eval()

# ----- load one MNIST sample -----
img, y = test[1]  # img: (1,28,28)

with torch.no_grad():
    out_mlp = model(img).squeeze(0)
    out_man = forward_manual(
        img.view(-1), Kl=K_matrix_list, n0=input_dim, n_list=mlp_hidden_layers
    ).squeeze(1)

print("max|diff|:", (out_mlp - out_man).abs().max().item())
print("allclose:", torch.allclose(out_mlp, out_man, atol=1e-4, rtol=1e-4))
print("mlp logits:", out_mlp)
print("manual logits:", out_man)

target_list = get_t_list(model, img, num_layers=len(wb))
RSO_list, shrinkage = compute_RSO_and_v(
    img.view(-1), K_matrix_list, input_dim, mlp_hidden_layers, target_list
)

# original MLP output
with torch.no_grad():
    out_mlp = model(img).squeeze(0)  # (10,)

# no-relu forward output (needs RSO,v computed for this same sample)
out_nr = forward_no_relu(
    img,
    K_matrix_list,
    RSO_list,
    shrinkage,
    n0=input_dim,
    n_list=mlp_hidden_layers,
).squeeze(1)


print("----------------")
print("max|diff|:", (out_mlp - out_nr).abs().max().item())
print("allclose:", torch.allclose(out_mlp, out_nr, atol=1e-4, rtol=1e-4))
print("mlp logits:", out_mlp)
print("no-relu logits:", out_nr)


max|diff|: 1.4066696166992188e-05
allclose: True
mlp logits: tensor([ -7.9487,   5.2788,  11.1139,   4.6059,  -6.8740,  -2.8400,  -4.9001,
          0.5340,   0.8711, -11.7324])
manual logits: tensor([ -7.9487,   5.2788,  11.1139,   4.6059,  -6.8740,  -2.8400,  -4.9001,
          0.5340,   0.8711, -11.7324])
----------------
max|diff|: 3.337860107421875e-05
allclose: True
mlp logits: tensor([ -7.9487,   5.2788,  11.1139,   4.6059,  -6.8740,  -2.8400,  -4.9001,
          0.5340,   0.8711, -11.7324])
no-relu logits: tensor([ -7.9487,   5.2788,  11.1140,   4.6059,  -6.8740,  -2.8400,  -4.9001,
          0.5340,   0.8711, -11.7324], grad_fn=<SqueezeBackward1>)


In [8]:
# x: (batch, n0) or (n0,)
def psi_x(x, L, N, n0):
    x = x.view(-1, n0)  # (B, n0)
    B = x.size(0)
    ones = torch.ones(B, L, device=x.device, dtype=x.dtype)
    zeros = torch.zeros(B, N - n0 - L, device=x.device, dtype=x.dtype)
    return torch.cat([x, ones, zeros], dim=1).T


# RSO[l]: (p_l,p_l), v[l]: scalar, N: int
def make_Rhat(RSO, v, N):
    Rhat = []
    for R, s in zip(RSO, v):
        p = R.size(0)
        Rhat.append(
            torch.block_diag((s * R), torch.eye(N - p, device=R.device, dtype=R.dtype))
        )
    # append identity as the last Rhat (e.g., R3)
    Rhat.append(torch.eye(N, device=Rhat[0].device, dtype=Rhat[0].dtype))
    return Rhat  # list of N×N


def make_K_theta(K_list, dtype=torch.float64):
    dev = K_list[0].device
    N = K_list[0].shape[0]
    K_theta = torch.eye(N, dtype=dtype, device=dev)
    for K in K_list:
        K_theta = K.to(dtype=dtype) @ K_theta  # cast K to match dtype
    return K_theta


def build_phi(Rhat, Kl, K_inv_list):
    # Kl: [K1,...,KL], Rhat: [R1,...,RL-1]
    L = len(Kl)

    phi = Rhat[L - 1]  # start with R_L
    for l in range(L, 1, -1):  # l = L, L-1, ..., 2
        phi = phi @ Kl[l - 1]  # ... @ K_l
        phi = phi @ Rhat[l - 2]  # ... @ R_{l-1}

    # inverse tail: K2^{-1} ... KL^{-1}
    for l in range(2, L + 1):
        phi = phi @ K_inv_list[l - 1]

    return phi


def forward_phi(x, phi, K_theta, L, n0=784, dtype=torch.float64):
    dev = phi.device
    N = K_theta.shape[0]

    psi_vec = psi_x(x.to(device=dev, dtype=dtype), L=L, N=N, n0=n0).to(dev, dtype)
    phi = phi.to(device=dev, dtype=dtype)
    K_theta = K_theta.to(device=dev, dtype=dtype)

    with torch.no_grad():
        y_tilde = phi @ (K_theta @ psi_vec)  # (N,1)
    return y_tilde[:10]


In [9]:
Rhat = make_Rhat(RSO_list, shrinkage, embedding_dim)
K_matrix_inv_list = [torch.linalg.inv(K) for K in K_matrix_list]
K_theta = make_K_theta(K_matrix_list, dtype=torch.float64)
phi = build_phi(Rhat, K_matrix_list, K_matrix_inv_list)

# ----- MLP logits -----
with torch.no_grad():
    out_mlp = model(img).squeeze(0)

# ----- phi-based logits -----
# forward_phi returns (10,1) in your implementation, so squeeze to (10,)
out_phi = forward_phi(img.view(-1), phi, K_theta, L=len(wb), n0=input_dim).squeeze(1)

# ----- compare safely (match dtypes) -----
out_phi_f = out_phi.to(dtype=out_mlp.dtype)  # cast to float32 to match out_mlp

print("label:", y)
print("mlp dtype:", out_mlp.dtype, "| phi dtype:", out_phi.dtype)
print("max|diff|:", (out_mlp - out_phi_f).abs().max().item())
print("allclose:", torch.allclose(out_mlp, out_phi_f, atol=1e-4, rtol=1e-4))
print("mlp logits:", out_mlp)
print("phi logits:", out_phi_f)


label: 2
mlp dtype: torch.float32 | phi dtype: torch.float64
max|diff|: 0.00027751922607421875
allclose: False
mlp logits: tensor([ -7.9487,   5.2788,  11.1139,   4.6059,  -6.8740,  -2.8400,  -4.9001,
          0.5340,   0.8711, -11.7324])
phi logits: tensor([ -7.9487,   5.2788,  11.1140,   4.6059,  -6.8741,  -2.8399,  -4.9000,
          0.5337,   0.8711, -11.7326])


In [10]:
# import time


# def time_avg(fn, N=100, sync_cuda=True):
#     # warmup
#     for _ in range(10):
#         fn()
#     if sync_cuda and torch.cuda.is_available():
#         torch.cuda.synchronize()

#     t0 = time.perf_counter()
#     for _ in range(N):
#         fn()
#     if sync_cuda and torch.cuda.is_available():
#         torch.cuda.synchronize()
#     t1 = time.perf_counter()

#     return (t1 - t0) / N


# N = 200  # <-- set N here (number of forward passes to average)

# mlp_time = time_avg(lambda: model(img.unsqueeze(0)).squeeze(0), N=N)
# phi_time = time_avg(lambda: forward_phi(x, phi, K_theta, L=3, n0=784).squeeze(1), N=N)

# print(f"MLP avg forward over N={N}: {mlp_time * 1e3:.3f} ms")
# print(f"Phi avg forward over N={N}: {phi_time * 1e3:.3f} ms")


In [11]:
# ------------------------------
# 1) Build dataset: (x -> phi_x)
# ------------------------------
@torch.no_grad()
def compute_phi_from_img(img: torch.Tensor) -> torch.Tensor:
    """
    img: (1,28,28) on device
    returns phi_x: (N, N) on device
    """
    x = img.view(-1)  # (784,)
    target_list = get_t_list(model, img, num_layers)  # assumes img is on device
    RSO_list, shrinkage = compute_RSO_and_v(
        x, K_matrix_list, n0=input_dim, n_list=mlp_hidden_layers, t_list=target_list
    )
    Rhat_list = make_Rhat(RSO_list, shrinkage, embedding_dim)
    phi_x = build_phi(Rhat_list, K_matrix_list, K_matrix_inv_list)  # (N, N)
    return phi_x


def build_phi_dataset_from_mnist(mnist_ds, num_per_class=1000):
    """
    Precomputes a balanced dataset across classes.
    Stores X and Y on CPU for memory efficiency.
      X: (M, 784)
      Y: (M, N*N)  (flattened phi_x)
    """
    model.eval()

    counts = [0] * 10
    X_list, Y_list = [], []

    for img, label in mnist_ds:
        c = int(label)
        if counts[c] >= num_per_class:
            continue

        # img = img.to(device)
        phi_x = compute_phi_from_img(img)  # (N,N) on device

        x_cpu = img.view(-1).detach().float().cpu()  # (784,)
        phi_cpu = phi_x.detach().to(torch.float32).reshape(-1).cpu()  # (N*N,)

        X_list.append(x_cpu)
        Y_list.append(phi_cpu)
        counts[c] += 1

        if all(cnt >= num_per_class for cnt in counts):
            break

    X = torch.stack(X_list, dim=0)  # (M, 784)
    Y = torch.stack(Y_list, dim=0)  # (M, N*N)
    return X, Y, counts


X, Y, counts = build_phi_dataset_from_mnist(train)
print("Counts per class:", counts)
print("X shape:", X.shape, "Y shape:", Y.shape)


Counts per class: [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
X shape: torch.Size([10000, 784]) Y shape: torch.Size([10000, 624100])


In [12]:
class PhiRegressionDataset(Dataset):
    def __init__(self, X_cpu: torch.Tensor, Y_cpu: torch.Tensor):
        self.X = X_cpu
        self.Y = Y_cpu

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


full_ds = PhiRegressionDataset(X, Y)

# Train/val split
val_frac = 0.15
n_val = int(len(full_ds) * val_frac)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(0)
)

batch_size = 512
train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
)

In [18]:
# ------------------------------
# 2) 1-layer "MLP": Linear only
# ------------------------------
class OneLayerMLP(nn.Module):
    def __init__(self, in_dim=784, out_dim=None):
        super().__init__()
        assert out_dim is not None
        self.fc = nn.Linear(in_dim, in_dim)
        self.fc2 = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        # x: (B, 784)
        x = self.fc2(x)
        return x


device = "cuda"
out_dim = embedding_dim**2
mlp = OneLayerMLP(in_dim=784, out_dim=out_dim).to(device)


# ------------------------------
# 3) Train (regression to phi_x)
# ------------------------------
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_n = 0

    for xb_cpu, yb_cpu in loader:
        xb = xb_cpu.to(device, non_blocking=True)
        yb = yb_cpu.to(device, non_blocking=True)

        pred = model(xb)  # (B, N*N)
        loss = F.mse_loss(pred, yb)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(1, total_n)


lr = 3e-4
optimizer = torch.optim.Adam(mlp.parameters(), lr=lr)


epochs = 5
for epoch in range(1, epochs + 1):
    train_loss = run_epoch(mlp, train_loader, optimizer=optimizer)
    val_loss = run_epoch(mlp, val_loader, optimizer=None)
    print(f"Epoch {epoch:02d} | train MSE: {train_loss:.6e} | val MSE: {val_loss:.6e}")


# ------------------------------
# 4) Using the trained MLP
# ------------------------------
@torch.no_grad()
@torch.no_grad()
def predict_phi_from_x_batch(x_flat: torch.Tensor) -> torch.Tensor:
    # x_flat: (B,784) or (784,)
    if x_flat.dim() == 1:
        x_flat = x_flat.unsqueeze(0)
    x_flat = x_flat.to(device)
    y = mlp(x_flat)  # (B, N*N)
    return y.view(-1, embedding_dim, embedding_dim)  # (B, N, N)


# Example:
img0, _ = train[0]
phi_pred = predict_phi_from_x_batch(img0.view(-1))
print("phi_pred shape:", phi_pred.shape)  # (N, N)

Epoch 01 | train MSE: 7.809997e+03 | val MSE: 7.720234e+03
Epoch 02 | train MSE: 7.808082e+03 | val MSE: 7.718310e+03
Epoch 03 | train MSE: 7.806178e+03 | val MSE: 7.716398e+03
Epoch 04 | train MSE: 7.804285e+03 | val MSE: 7.714489e+03
Epoch 05 | train MSE: 7.802396e+03 | val MSE: 7.712587e+03
phi_pred shape: torch.Size([1, 790, 790])


In [19]:
@torch.no_grad()
def phi_from_mlp_batch(x_flat: torch.Tensor) -> torch.Tensor:
    """
    x_flat: (B, 784) or (784,)
    returns: (B, N, N) (or (1,N,N) if single) on device, dtype=K_theta.dtype
    """
    if x_flat.dim() == 1:
        x_flat = x_flat.unsqueeze(0)  # (1,784)
    x_flat = x_flat.to(device)

    phi_flat = mlp(x_flat)  # (B, N*N)
    phi = phi_flat.view(-1, embedding_dim, embedding_dim)
    return phi


@torch.no_grad()
def forward_phi_batch(x, phi, K_theta, L, n0=784, dtype=torch.float64):
    """
    x  : (B, n0) or (n0,)
    phi: (B, N, N)  OR (N, N) (shared)
    returns logits: (B, 10)
    """
    dev = K_theta.device
    if x.dim() == 1:
        x = x.unsqueeze(0)  # (1,n0)
    x = x.to(device=dev, dtype=dtype)

    # psi_x returns (N, B)
    psi = psi_x(x, L=L, N=K_theta.shape[0], n0=n0).to(dev, dtype)  # (N,B)
    Kpsi = K_theta.to(dtype=dtype) @ psi  # (N,B)

    # Apply phi
    if phi.dim() == 2:
        # shared phi: (N,N) @ (N,B) -> (N,B)
        y = (phi.to(dev, dtype) @ Kpsi).T  # (B,N)
    else:
        # per-sample phi: bmm( (B,N,N), (B,N,1) ) -> (B,N,1) -> (B,N)
        phi = phi.to(dev, dtype)  # (B,N,N)
        Kpsi_vec = Kpsi.T.unsqueeze(-1)  # (B,N,1)
        y = torch.bmm(phi, Kpsi_vec).squeeze(-1)  # (B,N)

    return y[:, :10]  # (B,10)

In [20]:
# Assumes: mlp, forward_phi/psi_x exist (we'll add batched forward below),
#          K_theta, N, L, n0, device, test

mlp.eval()

# ---- pre-cast once (avoid doing this every call) ----
K_theta_b = K_theta.to(device=device, dtype=K_theta.dtype)


# ----- Batched test accuracy -----
batch_size = 1024
loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=0)

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)
        x = imgs.view(imgs.size(0), -1)  # (B,784)
        labels = labels.to(device)

        phi_hat = phi_from_mlp_batch(x)  # (B,N,N)
        logits = forward_phi_batch(
            x, phi_hat, K_theta_b, L=len(wb), n0=input_dim, dtype=K_theta_b.dtype
        )  # (B,10)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()

print(f"Test accuracy (batched phi-MLP + forward_phi): {correct / total:.4f}")


Test accuracy (batched phi-MLP + forward_phi): 0.3334


In [21]:
# from torch.utils.data import Subset

# # assumes: test (MNIST), device, model (pretrained), mlp (phi-MLP),
# #          forward_phi_batch (batched version), K_theta, N, L, n0

# mlp.eval()
# model.eval()

# # ---- settings ----
# B = 200  # SAME batch size for both paths
# num_samples = 200
# idx = list(range(num_samples))
# loader = DataLoader(Subset(test, idx), batch_size=B, shuffle=False)


# def sync():
#     if device.type == "cuda":
#         torch.cuda.synchronize()


# @torch.no_grad()
# def phi_from_mlp_batch(x_flat):  # x_flat: (B,784)
#     phi_flat = mlp(x_flat)  # (B, N*N)
#     return phi_flat.view(-1, embedding_dim, embedding_dim).to(K_theta.dtype)  # (B,N,N)


# # -------- warmup --------
# with torch.no_grad():
#     for i, (imgs, _) in enumerate(loader):
#         if i >= 2:
#             break
#         imgs = imgs.to(device)
#         x = imgs.view(imgs.size(0), -1)  # (B,784)
#         phi = phi_from_mlp_batch(x)
#         _ = forward_phi_batch(
#             x, phi, K_theta, L=L, n0=n0, dtype=K_theta.dtype
#         )  # (B,10)
#         _ = model(imgs)  # imgs already (B,1,28,28)
# sync()

# # -------- time: phi-MLP + forward_phi_batch --------
# t0 = time.perf_counter()
# with torch.no_grad():
#     for imgs, _ in loader:
#         imgs = imgs.to(device)
#         x = imgs.view(imgs.size(0), -1)
#         phi = phi_from_mlp_batch(x)
#         _ = forward_phi_batch(x, phi, K_theta, L=L, n0=n0, dtype=K_theta.dtype)
# sync()
# t1 = time.perf_counter()

# # -------- time: pretrained model forward --------
# t2 = time.perf_counter()
# with torch.no_grad():
#     for imgs, _ in loader:
#         imgs = imgs.to(device)
#         _ = model(imgs)
# sync()
# t3 = time.perf_counter()

# phi_path_total = t1 - t0
# orig_total = t3 - t2
# n = num_samples

# print(f"batch_size={B}, n={n}")
# print(
#     f"phi-MLP + forward_phi: total {phi_path_total:.6f}s | per-sample {phi_path_total / n * 1e3:.3f} ms"
# )
# print(
#     f"pretrained model fwd : total {orig_total:.6f}s | per-sample {orig_total / n * 1e3:.3f} ms"
# )
